# 03 — Drying Stage Explorer

Spray is stopped; hot air continues to remove **residual acetone** from the coated particles.
The coating mass is fixed at its end-of-spraying value.

**Move the sliders** to explore how post-spray drying conditions affect:
- Acetone removal rate from particles
- Product temperature during drying


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from fluid_bed.parameters import ProcessParameters
from fluid_bed.models.drying_stage import run_drying

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

In [ ]:
FIXED = dict(
    particle_density = 1000.0,
    cp_particle      = 1400.0,
    diameter_bed     = 0.60,
    rho_air          = 1.10,
    cp_air           = 1010.0,
)
print("Fixed properties loaded.")


In [ ]:
S = dict(continuous_update=False)

_controls03 = {
    "T_inlet_C":    widgets.FloatSlider(70.0, min=40,  max=100, step=5,   description="T inlet (°C)",    **S),
    "flow_m3h":     widgets.FloatSlider(300,  min=100, max=600, step=25,  description="Air flow (m³/h)", **S),
    "Y_init_pct":   widgets.FloatSlider(5.0,  min=0.5, max=20,  step=0.5, description="Y₀ acetone (%)", **S),
    "T_init_C":     widgets.FloatSlider(45.0, min=20,  max=75,  step=5,   description="T₀ product (°C)", **S),
    "duration_min": widgets.FloatSlider(15.0, min=2,   max=60,  step=2,   description="Duration (min)",  **S),
    "batch_kg":     widgets.FloatSlider(2.0,  min=0.5, max=5.0, step=0.5, description="Batch (kg)",      **S),
    "log_y_scale":  widgets.Checkbox(value=False, description="Log Y axis"),
}


def _run03(T_inlet_C, flow_m3h, Y_init_pct, T_init_C,
           duration_min, batch_kg, log_y_scale):
    m_air     = (flow_m3h / 3600.0) * FIXED["rho_air"]
    T_inlet_K = T_inlet_C + 273.15
    T_init_K  = T_init_C  + 273.15
    Y_init    = Y_init_pct / 100.0

    params = ProcessParameters(
        **FIXED,
        diameter_eq      = 200e-6,
        batch_size       = batch_kg,
        air_flow_rates   = (m_air, m_air, m_air),
        air_temperatures = (T_inlet_K, T_inlet_K, T_inlet_K),
    )

    res = run_drying(params, duration=duration_min * 60.0,
                     Y_particle_init=Y_init, T_particle_init=T_init_K)
    t_min       = res.t / 60.0
    T_p_C       = res.T_particle - 273.15
    T_g_C       = res.T_gas      - 273.15
    acetone_pct = res.Y_particle * 100.0

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    ax = axes[0]
    y_vals = np.maximum(acetone_pct, 1e-6)
    if log_y_scale:
        ax.semilogy(t_min, y_vals, color="darkorange", lw=2.5)
        ax.set_ylabel("Acetone on particles (wt %, log)")
    else:
        ax.plot(t_min, y_vals, color="darkorange", lw=2.5)
        ax.set_ylabel("Acetone on particles (wt %)")
    ax.set_xlabel("Time (min)"); ax.set_title("Particle drying"); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(t_min, T_g_C, "r--", lw=1.5, label="Gas T_g")
    ax.plot(t_min, T_p_C, "b-",  lw=2.5, label="Product T_p")
    ax.axhline(T_inlet_C, color="r", ls=":", alpha=0.35,
               label=f"Inlet setpoint {T_inlet_C:.0f} °C")
    ax.set_xlabel("Time (min)"); ax.set_ylabel("Temperature (°C)")
    ax.set_title("Temperatures during drying"); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close()
    print(f"Initial acetone : {Y_init_pct:.2f} %  |  "
          f"Final acetone : {acetone_pct[-1]:.4f} %  |  "
          f"Final T_product : {T_p_C[-1]:.1f} °C")


_out03 = widgets.interactive_output(_run03, _controls03)
clear_output(wait=True)
display(widgets.VBox(list(_controls03.values())), _out03)